In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math

In [4]:
df = pd.read_csv("Coursera.csv")

df = df.head(700)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("")
    else:
        df[col] = df[col].fillna(0)

df.head()

,partner,course,skills,rating,reviewcount,level,certificatetype,duration,crediteligibility
0,Google,Google Cybersecurity,"{"" Network Security"","" Python Programming"","" L...",4.8,16.4k,Beginner,Professional Certificate,3 - 6 Months,False
1,Google,Google Data Analytics,"{"" Data Analysis"","" R Programming"","" SQL"","" Bu...",4.8,133.4k,Beginner,Professional Certificate,3 - 6 Months,True
2,Google,Google Project Management:,"{"" Project Management"","" Strategy and Operatio...",4.8,97.3k,Beginner,Professional Certificate,3 - 6 Months,True
3,Google,Google Digital Marketing & E-commerce,"{"" Digital Marketing"","" Marketing"","" Marketing...",4.8,21.4k,Beginner,Professional Certificate,3 - 6 Months,False
4,Google,Google IT Support,"{"" Computer Networking"","" Network Architecture...",4.8,181.4k,Beginner,Professional Certificate,3 - 6 Months,True


In [5]:
df["combined_features"] = (
    df["course"].astype(str) + " " +
    df["skills"].astype(str) + " " +
    df["level"].astype(str) + " " +
    df["partner"].astype(str)
)

In [6]:
tfidf = TfidfVectorizer(stop_words="english")

matrix = tfidf.fit_transform(df["combined_features"])

similarity = cosine_similarity(matrix)

In [12]:
def recommend(course_name, top_n=10):

    course_name = course_name.strip().lower()

    matches = df[df["course"].str.lower().str.strip() == course_name]

    if matches.empty:
        print("Course not found!")
        return []

    index = matches.index[0]

    scores = list(enumerate(similarity[index]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i in scores[1:top_n+1]:
        recommendations.append(i[0])

    return recommendations

In [17]:
course = input("Enter Course Name: ")

recommendations = recommend(course)
target_index = df[df["course"].str.lower() == course.lower()].index[0]


for idx in recommendations:
    print(df.iloc[idx]["course"])

Advanced Learning Algorithms
Unsupervised Learning, Recommenders, Reinforcement Learning
Introduction to TensorFlow for Artificial Intelligence, Machine Learning, and Deep Learning
Machine Learning with Python
IBM Machine Learning
Supervised Machine Learning: Regression and Classification
Introduction to Artificial Intelligence (AI)
Convolutional Neural Networks
Reinforcement Learning
Convolutional Neural Networks in TensorFlow


In [19]:
print(df["course"].head(20))

0                              Google Cybersecurity
1                             Google Data Analytics
2                        Google Project Management:
3             Google Digital Marketing & E-commerce
4                                 Google IT Support
5                                  IBM Data Science
6                                  Google UX Design
7                                  IBM Data Analyst
8                                  Machine Learning
9                      Introduction to Data Science
10                       Generative AI for Everyone
11              IBM DevOps and Software Engineering
12                IBM Full Stack Software Developer
13                    Key Technologies for Business
14    Data Science Fundamentals with Python and SQL
15                    IBM & Darden Digital Strategy
16                             IBM Data Engineering
17              IBM Data Analytics with Excel and R
18      Data Analysis and Visualization Foundations
19          

In [22]:
target_index, recommendations = recommend("Machine Learning")

for idx in recommendations:
    print(df.iloc[idx]["course"])

ValueError: too many values to unpack (expected 2, got 10)

In [ ]:
def evaluate_metrics(relevance, total_relevant):

    k = len(relevance)

    precision = sum(relevance) / k

    recall = sum(relevance) / total_relevant

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    hits = 0
    ap_sum = 0

    for rank, rel in enumerate(relevance, start=1):
        if rel == 1:
            hits += 1
            ap_sum += hits / rank

    map_score = ap_sum / hits if hits > 0 else 0

    dcg = 0

    for rank, rel in enumerate(relevance, start=1):
        dcg += rel / math.log2(rank + 1)

    ideal = sorted(relevance, reverse=True)

    idcg = 0

    for rank, rel in enumerate(ideal, start=1):
        idcg += rel / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0

    return {
        "Precision@10": round(precision, 3),
        "Recall@10": round(recall, 3),
        "F1-Score@10": round(f1, 3),
        "MAP@10": round(map_score, 3),
        "NDCG@10": round(ndcg, 3)
    }

In [ ]:
target_skills = set(
    str(df.iloc[target_index]["skills"]).lower().split(",")
)

relevance = []

for idx in recommendations:

    rec_skills = set(
        str(df.iloc[idx]["skills"]).lower().split(",")
    )

    overlap = len(target_skills & rec_skills)

    if overlap >= 1:
        relevance.append(1)
    else:
        relevance.append(0)

print(relevance)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
total_relevant = max(sum(relevance), 1)

metrics = evaluate_metrics(
    relevance,
    total_relevant
)

metrics

{'Precision@10': 1.0,
 'Recall@10': 1.0,
 'F1-Score@10': 1.0,
 'MAP@10': 1.0,
 'NDCG@10': 1.0}